# Laborator 4 — Aplicații ale rețelei neuronale recurente (RNN)

#### Corețchi Mihai
#### Set de date: perechi de propoziții RO–EN (Tatoeba / Anki — `manythings.org/anki/ron-eng.zip`)
#### Obiectiv: identificarea limbii (**română** / **engleză**) a unei propoziții folosind o rețea neuronală recurentă (RNN), cu comparație față de **LSTM** și **GRU** (variantele uni- și **Bidirectional**).

## 0. Instalarea dependențelor

Această celulă instalează bibliotecile necesare în mediul curent, astfel încât notebook-ul să ruleze **în orice mediu** (Google Colab, Jupyter local, Kaggle etc.). În Colab majoritatea sunt deja prezente; rularea este idempotentă.

In [ ]:
%pip install -q tensorflow pandas numpy matplotlib seaborn scikit-learn

## 1. Importul bibliotecilor și configurarea mediului

In [ ]:
import os
import io
import time
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input, TextVectorization, Embedding,
    RNN, SimpleRNNCell, LSTM, GRU, Bidirectional, Dense,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report, f1_score,
)

%matplotlib inline
sns.set_theme(style="whitegrid")
tf.keras.utils.set_random_seed(101)

print("TensorFlow:", tf.__version__)

## 2. Obținerea și analiza exploratorie a datelor

Folosim corpusul **Tatoeba / Anki** de perechi de propoziții engleză–română (`ron-eng.zip` de pe `manythings.org`). Fiecare linie din `ron.txt` are forma `propoziție_EN  \t  propoziție_RO  \t  atribuire CC-BY`.

Din fiecare pereche construim **două exemple**: propoziția engleză (etichetă `0`) și cea românească (etichetă `1`). Rezultă un set **perfect echilibrat** (același număr de propoziții pe limbă), potrivit pentru o clasificare binară cu ieșire `sigmoid`.

In [ ]:
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
ron_txt = os.path.join(DATA_DIR, "ron.txt")

# manythings.org respinge cererile fără antet de browser (HTTP 406/403),
# de aceea trimitem un set complet de antete (User-Agent, Accept, Referer).
if not os.path.exists(ron_txt):
    url = "http://www.manythings.org/anki/ron-eng.zip"
    headers = {
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/120.0 Safari/537.36"),
        "Accept": ("text/html,application/xhtml+xml,application/xml;"
                   "q=0.9,image/avif,image/webp,*/*;q=0.8"),
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.manythings.org/anki/",
    }
    req = urllib.request.Request(url, headers=headers)
    blob = urllib.request.urlopen(req, timeout=120).read()
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        z.extract("ron.txt", DATA_DIR)
    print("Descărcat și extras:", ron_txt)
else:
    print("Fișier deja prezent:", ron_txt)

# Dacă descărcarea automată este blocată de rețea: descărcați manual
# http://www.manythings.org/anki/ron-eng.zip și puneți `ron.txt` în folderul `data/`.

In [ ]:
# Parsare: col. 0 = EN (eticheta 0), col. 1 = RO (eticheta 1)
LANGS = ["Engleză", "Română"]
texts, labels = [], []
with open(ron_txt, encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) < 2:
            continue
        en, ro = parts[0].strip(), parts[1].strip()
        if en and ro:
            texts.append(en); labels.append(0)
            texts.append(ro); labels.append(1)

df = pd.DataFrame({"text": texts, "label": labels})
df["lang"] = df["label"].map(lambda i: LANGS[i])
df["n_words"] = df["text"].str.split().str.len()

print("Total propoziții:", len(df))
print(df["lang"].value_counts())
df.sample(5, random_state=101)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.countplot(data=df, x="lang", ax=axes[0])
axes[0].set_title("Număr de propoziții per limbă (set echilibrat)")
axes[0].set_xlabel(""); axes[0].set_ylabel("Propoziții")

sns.histplot(data=df, x="n_words", hue="lang", bins=30,
             element="step", ax=axes[1])
axes[1].set_title("Distribuția lungimii propozițiilor")
axes[1].set_xlabel("Cuvinte / propoziție"); axes[1].set_ylabel("Frecvență")

plt.tight_layout()
plt.show()

In [ ]:
for lang in LANGS:
    print(f"\n— Exemple «{lang}» —")
    for s in df.loc[df["lang"] == lang, "text"].sample(5, random_state=7):
        print("  •", s)

## 3. Pregătirea datelor

- **Split stratificat 70/15/15** cu `sklearn.model_selection.train_test_split` (aplicat de două ori, `random_state=101`, `stratify=y`) → fiecare subset păstrează proporția egală română/engleză.
- Construim un pipeline `tf.data` cu `prefetch(AUTOTUNE)` (suprapune citirea datelor cu calculul), exact ca la laboratoarele anterioare. Textul intră ca șir brut de formă `(batch, 1)` — vectorizarea se face **în interiorul modelului**.
- `TextVectorization` transformă textul în secvențe de indici întregi (`output_mode="int"`), trunchiate/completate la `MAX_LEN`. **Adaptăm vocabularul doar pe textele de antrenare** (română + engleză împreună) → rezultă un **vocabular comun bilingv**, exact cerința obiectivului 1. Standardizarea implicită păstrează diacriticele românești (`ă â î ș ț`), semnal puternic pentru limbă.

In [ ]:
# Split stratificat 70/15/15 cu scikit-learn (train_test_split aplicat de 2 ori)
X = df["text"].to_numpy()
y = df["label"].to_numpy()

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=101, stratify=y,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=101, stratify=y_tmp,
)

splits = {
    "train": (X_train, y_train),
    "val":   (X_val,   y_val),
    "test":  (X_test,  y_test),
}
for name, (xs, ys) in splits.items():
    print(f"{name:5s}: {len(xs):6d} propoziții  "
          f"(EN={int((ys == 0).sum())}, RO={int((ys == 1).sum())})")

BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(xs, ys, shuffle=False):
    # forma (N, 1) — potrivită pentru Input(shape=(1,), dtype="string")
    ds = tf.data.Dataset.from_tensor_slices((xs.reshape(-1, 1), ys))
    if shuffle:
        ds = ds.shuffle(len(xs), seed=101)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(*splits["train"], shuffle=True)
val_ds = make_ds(*splits["val"])
test_ds = make_ds(*splits["test"])

In [ ]:
MAX_TOKENS = 20000
MAX_LEN = 50

vectorize_layer = TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=MAX_LEN,
)
# Adaptăm DOAR pe textele de antrenare (RO + EN) → vocabular comun bilingv.
vectorize_layer.adapt(splits["train"][0])

vocab = vectorize_layer.get_vocabulary()
print("Mărime vocabular:", len(vocab))
print("Primii 30 tokeni:", vocab[2:32])

In [ ]:
# Dovada vocabularului bilingv: tokeni clar românești ȘI clar englezești
ro_tokens = [w for w in vocab if any(c in w for c in "ăâîșțĂÂÎȘȚ")][:15]
en_tokens = [w for w in ["the", "and", "you", "what", "where", "is", "not", "have"]
             if w in vocab]
print("Tokeni clar românești (cu diacritice):", ro_tokens)
print("Tokeni clar englezești:                ", en_tokens)
print("→ Vocabularul este comun pentru ambele limbi (română + engleză).")

## 4. Crearea modelului RNN de bază (SimpleRNN)

Construim un model `Sequential` exact după scheletul din enunț:

- **`Input(shape=(1,), dtype="string")`** — modelul primește text brut; toată preprocesarea este integrată, deci predicția în timp real se face direct pe propoziții.
- **`TextVectorization`** (adaptat la pasul 3) → indici întregi.
- **`Embedding(MAX_TOKENS+1, 128)`** — fiecare token devine un vector dens de 128 dimensiuni, învățat odată cu rețeaua (cuvinte cu rol similar ajung apropiate în spațiu).
- **`RNN(SimpleRNNCell(64))`** — celula recurentă parcurge secvența și menține o **stare ascunsă** de 64 de unități care „rezumă" propoziția citită până la momentul curent.
- **`Dense(64, relu)`** apoi **`Dense(1, sigmoid)`** — capul de clasificare; ieșire o singură probabilitate (≈1 → română, ≈0 → engleză).

Funcția `build_model` reutilizează **același** `vectorize_layer` și aceeași structură, schimbând doar stratul recurent — astfel comparația dintre SimpleRNN / LSTM / GRU este corectă (singura variabilă este tipul de recurență).

In [ ]:
def build_model(recurrent_layer, name):
    # Model identic; variază doar stratul recurent. Reutilizează
    # vectorize_layer adaptat (vocabular bilingv comun).
    return Sequential([
        Input(shape=(1,), dtype="string"),
        vectorize_layer,
        Embedding(MAX_TOKENS + 1, 128, name="embedding"),
        recurrent_layer,
        Dense(64, activation="relu"),
        Dense(1, activation="sigmoid"),
    ], name=name)

rnn_model = build_model(
    RNN(SimpleRNNCell(64), return_sequences=False, return_state=False),
    name="SimpleRNN",
)
rnn_model.summary()

## 5. Compilarea și antrenarea modelului de bază

- **Optimizator:** `Adam(1e-3)` — robust, combină momentum și rate adaptive.
- **Loss:** `binary_crossentropy` — clasificare binară (engleză vs română) cu ieșire `sigmoid`.
- **Metrică:** `accuracy`.
- **`EarlyStopping`** pe `val_accuracy` cu `restore_best_weights=True` — oprește antrenarea când validarea nu mai progresează și păstrează cele mai bune greutăți (alege automat numărul optim de epoci).

Helper-ele `compile_fit` și `plot_history` vor fi refolosite pentru toate modelele comparate.

In [ ]:
def compile_fit(model, epochs=10):
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    early_stop = EarlyStopping(
        monitor="val_accuracy", patience=3,
        restore_best_weights=True, mode="max",
    )
    t0 = time.perf_counter()
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=epochs, callbacks=[early_stop], verbose=2,
    )
    return history, time.perf_counter() - t0

def plot_history(history, title):
    hist = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(hist["accuracy"], label="train accuracy")
    axes[0].plot(hist["val_accuracy"], label="val accuracy")
    axes[0].set_title(f"{title} — accuracy")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy"); axes[0].legend()

    axes[1].plot(hist["loss"], label="train loss")
    axes[1].plot(hist["val_loss"], label="val loss")
    axes[1].set_title(f"{title} — loss")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss"); axes[1].legend()

    plt.tight_layout()
    plt.show()

rnn_history, rnn_time = compile_fit(rnn_model)
print(f"Timp antrenare SimpleRNN: {rnn_time:.1f} s")
plot_history(rnn_history, "SimpleRNN")

## 6. Evaluarea modelului de bază

Evaluăm pe setul de **test** (nevăzut la antrenare): loss, acuratețe, F1 (macro/weighted), raport pe clase, matricea de confuzie și câteva predicții concrete pe propoziții din test.

In [ ]:
def evaluate_model(model):
    y_true = np.concatenate([yb.numpy() for _, yb in test_ds])
    y_prob = model.predict(test_ds, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
    return y_true, y_pred, y_prob

y_true, y_pred, _ = evaluate_model(rnn_model)
test_loss, test_acc = rnn_model.evaluate(test_ds, verbose=0)

print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(f"F1-macro:      {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"F1-weighted:   {f1_score(y_true, y_pred, average='weighted'):.4f}")
print()
print(classification_report(y_true, y_pred, target_names=LANGS, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LANGS, yticklabels=LANGS, cbar=False)
plt.title("Matricea de confuzie — SimpleRNN")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# Keras 3 nu acceptă direct array-uri NumPy de string-uri (object dtype);
# le convertim într-un tensor tf.string de formă (N, 1) — exact ce produce
# și pipeline-ul tf.data. Helper-ul e refolosit la predicția în timp real.
def to_input(texts):
    return tf.constant(np.asarray(texts, dtype=str).reshape(-1, 1))

# Predicții concrete pe 10 propoziții aleatorii din setul de test
sx, sy = splits["test"]
pick = np.random.default_rng(1).choice(len(sx), 10, replace=False)
preds = (rnn_model.predict(to_input(sx[pick]), verbose=0).ravel() >= 0.5).astype(int)

for i, p in zip(pick, preds):
    mark = "✓" if sy[i] == p else "✗"
    print(f"[{mark}] real={LANGS[sy[i]]:8s} pred={LANGS[p]:8s} | {sx[i][:70]}")

## 7. Comparație cu alte modele RNN

Antrenăm, în aceleași condiții (același vocabular, Embedding, cap dens, optimizator și `EarlyStopping`), patru variante recurente alternative. `SimpleRNN` suferă de **dispariția gradientului** pe secvențe lungi; `LSTM` și `GRU` folosesc **porți** care rețin/uită informație selectiv. Varianta **`Bidirectional`** citește propoziția în ambele sensuri (stânga→dreapta și dreapta→stânga), utilă fiindcă indicii de limbă pot apărea oriunde în propoziție.

### 7.1 LSTM

In [ ]:
lstm_model = build_model(LSTM(64), name="LSTM")
lstm_history, lstm_time = compile_fit(lstm_model)
print(f"Timp antrenare LSTM: {lstm_time:.1f} s")
plot_history(lstm_history, "LSTM")

### 7.2 Bidirectional LSTM

In [ ]:
bilstm_model = build_model(Bidirectional(LSTM(64)), name="BiLSTM")
bilstm_history, bilstm_time = compile_fit(bilstm_model)
print(f"Timp antrenare BiLSTM: {bilstm_time:.1f} s")
plot_history(bilstm_history, "Bidirectional LSTM")

### 7.3 GRU

In [ ]:
gru_model = build_model(GRU(64), name="GRU")
gru_history, gru_time = compile_fit(gru_model)
print(f"Timp antrenare GRU: {gru_time:.1f} s")
plot_history(gru_history, "GRU")

### 7.4 Bidirectional GRU

In [ ]:
bigru_model = build_model(Bidirectional(GRU(64)), name="BiGRU")
bigru_history, bigru_time = compile_fit(bigru_model)
print(f"Timp antrenare BiGRU: {bigru_time:.1f} s")
plot_history(bigru_history, "Bidirectional GRU")

## 8. Compararea modelelor

Sintetizăm într-un tabel numărul de parametri, cea mai bună acuratețe pe validare, acuratețea și F1 pe test, timpul de antrenare și numărul de epoci efectiv parcurse (cu `EarlyStopping`).

In [ ]:
def summarize(name, model, history, elapsed):
    yt, yp, _ = evaluate_model(model)
    return {
        "Model": name,
        "Parametri": model.count_params(),
        "Acc. validare": round(max(history.history["val_accuracy"]), 4),
        "Acc. test": round(accuracy_score(yt, yp), 4),
        "F1-weighted": round(f1_score(yt, yp, average="weighted"), 4),
        "Timp (s)": round(elapsed, 1),
        "Epoci": len(history.history["loss"]),
    }

results = pd.DataFrame([
    summarize("SimpleRNN", rnn_model,    rnn_history,    rnn_time),
    summarize("LSTM",      lstm_model,   lstm_history,   lstm_time),
    summarize("BiLSTM",    bilstm_model, bilstm_history, bilstm_time),
    summarize("GRU",       gru_model,    gru_history,    gru_time),
    summarize("BiGRU",     bigru_model,  bigru_history,  bigru_time),
])
results

In [ ]:
plt.figure(figsize=(9, 4))
ax = sns.barplot(data=results, x="Model", y="Acc. test")
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f")
plt.title("Acuratețea pe setul de test — comparație modele RNN")
plt.ylim(0.8, 1.005)
plt.xlabel(""); plt.ylabel("Acuratețe test")
plt.tight_layout()
plt.show()

## 9. Predicție în timp real (introducerea unei propoziții)

Conform enunțului, modelul trebuie să permită **testarea în timp real cu valori noi**. Folosim modelul cu cea mai bună acuratețe pe test. `predict_language` primește text brut (preprocesarea e integrată în model) și întoarce limba prezisă + încrederea.

Mai jos: o listă fixă de propoziții **noi** (care nu apar în set), în română și engleză — pentru capturi de ecran reproductibile — urmată de o buclă interactivă în care puteți introduce propriile propoziții (Enter gol = stop).

In [ ]:
best_name = results.loc[results["Acc. test"].idxmax(), "Model"]
models_by_name = {
    "SimpleRNN": rnn_model, "LSTM": lstm_model, "BiLSTM": bilstm_model,
    "GRU": gru_model, "BiGRU": bigru_model,
}
best_model = models_by_name[best_name]
print("Model folosit pentru predicție:", best_name)

def predict_language(text, model=None):
    model = model or best_model
    prob = float(model.predict(to_input([text]), verbose=0).ravel()[0])
    lang = LANGS[int(prob >= 0.5)]
    conf = prob if prob >= 0.5 else 1.0 - prob
    return lang, conf

new_sentences = [
    "Mâine plec la munte împreună cu prietenii mei.",
    "Îmi place foarte mult să citesc cărți de istorie.",
    "Unde ai pus cheile de la mașină?",
    "The weather is really nice today, let's go outside.",
    "I have never been to Japan but I would love to visit.",
    "Could you please tell me where the train station is?",
]
for s in new_sentences:
    lang, conf = predict_language(s)
    print(f"[{lang:8s} {conf * 100:5.1f}%]  {s}")

In [ ]:
# Testare interactivă „în timp real". Protejat cu try/except astfel încât
# rularea automată (fără stdin, ex. nbconvert) să nu se blocheze.
try:
    while True:
        s = input("Propoziție (Enter gol = stop): ").strip()
        if not s:
            break
        lang, conf = predict_language(s)
        print(f"  → {lang} (încredere {conf * 100:.1f}%)")
except (EOFError, KeyboardInterrupt):
    print("(intrare interactivă indisponibilă — pas omis)")

## 10. Concluzii

În această lucrare am construit, folosind **TensorFlow / `tf.keras`**, o **rețea neuronală recurentă (RNN)** care **identifică limba** (română vs engleză) a unei propoziții. Am parcurs tot fluxul cerut în enunț: obținerea datelor (corpus bilingv Tatoeba/Anki descărcat automat), analiza exploratorie (set echilibrat, distribuția lungimii propozițiilor), **preprocesarea limbajului natural** cu `TextVectorization` adaptat pe textele de antrenare — rezultând un **vocabular comun română + engleză** (obiectivul 1), proiectarea modelului de bază cu `SimpleRNNCell`, antrenarea cu `EarlyStopping` și evaluarea pe un set de test independent.

Am **comparat** apoi modelul de bază cu patru alternative — **LSTM**, **GRU** și variantele lor **Bidirectional**. SimpleRNN învață deja bine sarcina (limba este puternic separabilă: vocabular aproape disjunct, diacriticele românești `ă â î ș ț` și cuvinte funcționale frecvente — `the`, `and`, `și`, `este` — sunt indici foarte puternici), de aceea toate modelele ating o acuratețe ridicată. Totuși, **LSTM/GRU** sunt mai stabile la antrenare (porțile lor evită dispariția gradientului), iar variantele **Bidirectional** valorifică contextul din ambele sensuri ale propoziției, oferind de regulă cea mai bună acuratețe — cu prețul a aproximativ dublului de parametri și timp de antrenare.

Concluzia practică: pentru o sarcină „ușoară" și secvențe scurte, un RNN simplu este suficient și ieftin; pe măsură ce dependențele devin mai lungi și mai subtile (analiză de sentiment, traducere automată), **LSTM/GRU** și variantele **Bidirectional** devin alegerea corectă. Am exersat astfel întreg ciclul unei aplicații RNN de NLP: preprocesare → embedding → recurență → clasificare → evaluare → **predicție în timp real**.

## 11. Întrebări de autoevaluare

#### 1. Care sunt pașii de creare a unei rețele neuronale recurente (RNN)?

(1) **Definirea problemei** și colectarea unui set adecvat (aici: corpus bilingv RO–EN); (2) **preprocesarea textului** — curățare, tokenizare/vectorizare (`TextVectorization`), trunchiere/padding la o lungime comună, split train/val/test; (3) **stratul de embedding** care transformă indicii de tokeni în vectori denși; (4) **stratul recurent** (SimpleRNN/LSTM/GRU, eventual `Bidirectional`) care procesează secvența și produce o reprezentare; (5) **capul dens** cu funcția de activare potrivită la ieșire (`sigmoid` pentru binar); (6) **compilare** — loss + optimizator + metrici; (7) **antrenare** cu validare și `EarlyStopping`; (8) **evaluare** pe test (acuratețe, F1, matrice de confuzie); (9) **predicție în timp real** pe valori noi; (10) **optimizare/iterare**.

#### 2. Care sunt librăriile destinate procesării limbajului natural (NLP) în Python? Care sunt avantajele fiecărei librării?

- **TensorFlow / Keras** — straturi NLP integrate (`TextVectorization`, `Embedding`, `LSTM`, `GRU`), pipeline `tf.data`, antrenare GPU/TPU; ușor de pus în producție. (folosită aici)
- **PyTorch** — flexibil, dinamic, preferat în cercetare.
- **NLTK** — clasic, didactic: tokenizare, stemming, stopwords, corpora.
- **spaCy** — rapid, industrial: tokenizare, POS, NER, lematizare, pipeline-uri gata făcute.
- **Hugging Face Transformers** — modele pre-antrenate de ultimă generație (BERT, GPT) și fine-tuning simplu.
- **Gensim** — modelare de subiecte și embeddings (Word2Vec, FastText).
- **scikit-learn** — vectorizare clasică (TF-IDF) și modele de bază; folosită aici pentru metrici.

#### 3. Care sunt pașii de procesare a textului pentru a fi transformat în date de intrare în straturile Dense?

(1) **Normalizare/standardizare** (litere mici, eliminarea punctuației — diacriticele se păstrează); (2) **tokenizare** (împărțire în cuvinte); (3) **indexare** prin vocabular — fiecare token devine un întreg; (4) **padding/trunchiere** la o lungime fixă (`output_sequence_length`); (5) **embedding** — indicii devin vectori denși; (6) **stratul recurent** comprimă secvența de vectori într-un vector de stare de dimensiune fixă; (7) acest vector intră în **straturile `Dense`**. Pe scurt: text → indici → vectori → stare recurentă (vector fix) → `Dense`.

#### 4. Cum alegem numărul de straturi rnn? Dar numărul de straturi Dense într-o rețea RNN?

Numărul de **straturi recurente** depinde de complexitatea dependențelor: 1 strat este suficient pentru sarcini simple/secvențe scurte (cazul de față); 2–3 straturi „stivuite" (cu `return_sequences=True` între ele) pentru dependențe mai complexe — mai multe straturi cresc capacitatea, dar și riscul de supra-antrenare și costul. Pentru **straturile `Dense`** finale, 1–2 sunt de regulă suficiente (un strat ascuns + stratul de ieșire); puterea de modelare a secvenței vine din partea recurentă, nu din capul dens. Se pornește mic și se mărește doar dacă apare underfitting, monitorizând curbele de validare.

#### 5. De ce este necesară transformarea textului cu `tf.keras.layers.TextVectorization`?

Rețelele neuronale operează cu **numere**, nu cu șiruri de caractere. `TextVectorization` standardizează textul, îl tokenizează, construiește un **vocabular** (prin `adapt`) și transformă fiecare propoziție într-o secvență de **indici întregi** de lungime fixă (padding/trunchiere). Integrat în model, face ca modelul să primească text brut și să fie ușor de folosit în predicție în timp real.

#### 6. Ce este `tf.keras.layers.Embedding`?

Un strat care mapează fiecare **index de token** la un **vector dens** învățabil (aici 128 dimensiuni). Spre deosebire de one-hot (rar, fără relații), embedding-ul este **dens** și învățat odată cu rețeaua, astfel încât tokeni cu rol/sens similar ajung apropiați în spațiul vectorial. Practic, este un tabel de căutare `(MAX_TOKENS+1) × 128` optimizat prin backpropagation.

#### 7. Ce funcții de activare folosim în straturile interne, dar în stratul de ieșire?

În straturile **interne**: `ReLU` pentru straturile `Dense`; în celulele recurente, neliniarități interne `tanh` (și `sigmoid` pentru porțile LSTM/GRU). În stratul de **ieșire**: `sigmoid` pentru clasificare **binară** (cazul nostru — engleză vs română), `softmax` pentru multi-clasă, `linear` pentru regresie.

#### 8. Ce funcție este folosită la calcularea funcției loss?

Pentru clasificare **binară** folosim **binary cross-entropy**:

$$L = -\frac{1}{N}\sum_{i=1}^{N}\big[\,y_i \log(\hat{y}_i) + (1 - y_i)\log(1 - \hat{y}_i)\,\big]$$

unde $y_i \in \{0,1\}$ este eticheta reală (0 = engleză, 1 = română) și $\hat{y}_i$ probabilitatea prezisă de `sigmoid`. Penalizează puternic încrederea mare într-o predicție greșită. (Pentru multi-clasă s-ar folosi categorical / sparse categorical cross-entropy.)

#### 9. Care este diferența între rețeaua neuronală de tip LSTM și Bidirectional LSTM? Dați exemple concrete de aplicare.

Un **LSTM** citește secvența **într-o singură direcție** (stânga → dreapta); la fiecare pas „vede" doar contextul anterior. Un **Bidirectional LSTM** rulează două LSTM-uri — unul înainte, unul înapoi — și concatenează stările, deci fiecare poziție beneficiază de contextul **din ambele sensuri** (dublând parametrii). Exemple: **LSTM** unidirecțional — generare de text, predicție de serii temporale, autocomplete (viitorul nu e disponibil). **BiLSTM** — clasificare/etichetare când întreaga propoziție e disponibilă: NER, POS tagging, analiză de sentiment, identificarea limbii (cazul nostru — indicii pot apărea oriunde în propoziție).

#### 10. Care este diferența între rețeaua neuronală de tip GRU și Bidirectional GRU? Dați exemple concrete de aplicare.

**GRU** este o variantă de celulă cu porți (reset + update) mai simplă decât LSTM (fără stare de celulă separată), deci are mai puțini parametri și se antrenează mai rapid, cu performanță adesea comparabilă. **GRU** procesează secvența într-un singur sens; **Bidirectional GRU** o procesează în ambele sensuri și concatenează stările. Exemple: **GRU** unidirecțional — streaming/inferență în timp real, serii temporale, dispozitive cu resurse limitate (mai ieftin ca LSTM). **BiGRU** — clasificare de text, NER, identificarea limbii pe seturi mari unde contează atât viteza, cât și contextul bidirecțional.